# Decision Trees, Ensembles, and Regularization

Three ideas dominate modern tabular machine learning: **decision trees** that make predictions through interpretable sequences of questions, **ensemble methods** that combine many models into a stronger predictor, and **regularization** that prevents models from memorizing noise. Together, they form the backbone of practical machine learning on structured data.

Random forests and gradient boosting consistently win Kaggle competitions and power production systems in finance, healthcare, and e-commerce. Understanding how they work — and how regularization keeps them from overfitting — is essential for building reliable models.

This notebook goes deep into decision tree mechanics, ensemble strategies (bagging and boosting), feature importance, and the full spectrum of regularization techniques applied to both linear models and tree-based algorithms.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, make_classification, make_regression
from sklearn.model_selection import train_test_split, cross_val_score, validation_curve
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                                 BaggingClassifier, AdaBoostClassifier)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. Decision Trees in Depth

A decision tree partitions the feature space into rectangular regions through a sequence of binary splits. Each internal node tests a feature against a threshold; each branch represents the outcome; each leaf holds a prediction (class label or numeric value).

Trees are built **recursively**:

1. Start with all data at the root.
2. Find the feature and threshold that best split the data.
3. Create two child nodes and assign data points.
4. Repeat for each child until a stopping criterion is met.

The key question at every node: **which split best separates the classes (or reduces regression error)?**

### 1.1 Splitting Criteria for Classification

**Gini Impurity** — measures how often a randomly chosen sample would be incorrectly classified if labeled according to the class distribution in the node:

$$Gini = 1 - \sum_{k=1}^{K} p_k^2$$

where $p_k$ is the proportion of class $k$ in the node. Gini = 0 means pure node (all one class). Gini is maximized when classes are equally distributed.

**Entropy (Information Gain)** — measures the disorder or uncertainty in the node:

$$H = -\sum_{k=1}^{K} p_k \log_2(p_k)$$

A good split maximizes **information gain** — the reduction in entropy after splitting:

$$IG = H_{\text{parent}} - \left(\frac{n_L}{n} H_{\text{left}} + \frac{n_R}{n} H_{\text{right}}\right)$$

Gini and entropy usually produce similar trees. Gini is slightly faster to compute; entropy may produce slightly more balanced trees.

In [ ]:
# Gini and Entropy for different class distributions
def gini(probs):
    return 1 - np.sum(probs**2)

def entropy(probs):
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

distributions = {
    "Pure (100% A)": [1.0, 0.0],
    "Balanced (50/50)": [0.5, 0.5],
    "Skewed (90/10)": [0.9, 0.1],
    "Three-class equal": [1/3, 1/3, 1/3],
}

print(f"{'Distribution':20s} {'Gini':>8s} {'Entropy':>8s}")
print("-" * 38)
for name, probs in distributions.items():
    print(f"{name:20s} {gini(np.array(probs)):8.4f} {entropy(np.array(probs)):8.4f}")

### 1.2 Splitting Criteria for Regression

For regression trees, splits minimize the **variance** (or MSE) of the target within each child node:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \bar{y})^2$$

The prediction at each leaf is the **mean** of the target values in that leaf.

### 1.3 Strengths of Decision Trees

- **Interpretable** — the tree can be visualized and explained to non-technical stakeholders.
- **No feature scaling needed** — splits are based on thresholds, not distances.
- **Handles mixed types** — numerical and categorical features naturally.
- **Captures nonlinearity and interactions** — no manual feature engineering required.
- **Fast prediction** — traverse from root to leaf.

In [ ]:
# Decision tree visualization at different depths
cancer = load_breast_cancer()
X, y = cancer.data[:, :2], cancer.target  # 2 features for visualization
feature_names = cancer.feature_names[:2]
class_names = cancer.target_names

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, depth, title in zip(axes, [2, 4], ["max_depth=2", "max_depth=4"]):
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X, y)
    plot_tree(dt, feature_names=feature_names, class_names=class_names,
              filled=True, rounded=True, ax=ax, fontsize=9)
    acc = accuracy_score(y, dt.predict(X))
    ax.set_title(f"{title} (train accuracy={acc:.3f})")

plt.tight_layout()
plt.show()

---

## 2. The Overfitting Problem in Trees

An unconstrained decision tree grows until every leaf is pure — potentially one leaf per training sample. This perfectly fits training data but generalizes poorly.

**Controlling tree complexity** through hyperparameters:

| Hyperparameter | Effect |
|---------------|--------|
| `max_depth` | Maximum levels in the tree |
| `min_samples_split` | Minimum samples required to split a node |
| `min_samples_leaf` | Minimum samples required in a leaf |
| `max_leaf_nodes` | Maximum number of leaf nodes |
| `min_impurity_decrease` | Minimum impurity reduction to justify a split |

**Pruning** — grow a full tree, then remove branches that do not improve validation performance. Cost-complexity pruning (used in scikit-learn's `ccp_alpha` parameter) finds the optimal tradeoff between tree size and fit quality.

The goal is a tree that captures real patterns without memorizing noise.

In [ ]:
# Overfitting: tree depth vs train/test accuracy
X_full, y_full = cancer.data, cancer.target
X_tr, X_te, y_tr, y_te = train_test_split(X_full, y_full, test_size=0.3, random_state=42)

depths = range(1, 21)
train_accs, test_accs = [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_tr, y_tr)
    train_accs.append(accuracy_score(y_tr, dt.predict(X_tr)))
    test_accs.append(accuracy_score(y_te, dt.predict(X_te)))

plt.figure(figsize=(9, 5))
plt.plot(depths, train_accs, "b-o", label="Train", markersize=4)
plt.plot(depths, test_accs, "r-o", label="Test", markersize=4)
best_depth = list(depths)[np.argmax(test_accs)]
plt.axvline(best_depth, color="green", linestyle="--", label=f"Best depth={best_depth}")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Decision Tree: depth controls overfitting")
plt.legend()
plt.show()

---

## 3. Ensemble Learning

An **ensemble** combines multiple models (called **base learners** or **weak learners**) to produce a prediction more accurate than any individual model. The idea rests on the principle that a group of diverse, moderately accurate models can outperform a single highly tuned model — if their errors are uncorrelated.

Two fundamental ensemble strategies:

```
  BAGGING (parallel)              BOOSTING (sequential)
  ┌─────┐ ┌─────┐ ┌─────┐        ┌─────┐    ┌─────┐    ┌─────┐
  │Model│ │Model│ │Model│        │Model│ →  │Model│ →  │Model│
  │  1  │ │  2  │ │  3  │        │  1  │    │  2  │    │  3  │
  └──┬──┘ └──┬──┘ └──┬──┘        └──┬──┘    └──┬──┘    └──┬──┘
     └───────┼───────┘              └────────┼───────────┘
          Average/Vote                    Weighted combo
```

- **Bagging** — train models independently on random subsets, average predictions. Reduces **variance**.
- **Boosting** — train models sequentially, each correcting previous errors. Reduces **bias**.

---

## 4. Bagging and Random Forests

### 4.1 Bagging (Bootstrap Aggregating)

1. Create $B$ bootstrap samples (random sampling with replacement) from the training data.
2. Train a model on each bootstrap sample.
3. **Classification:** majority vote. **Regression:** average predictions.

Bagging reduces overfitting by averaging out the noise that individual models capture. It works best with high-variance models (deep decision trees) that overfit on their own.

### 4.2 Random Forest

Random Forest extends bagging with an additional randomization: at each split, only a **random subset of features** is considered. This decorrelates the trees — without it, all trees would split on the same dominant feature and the ensemble would gain little.

Key hyperparameters:
- `n_estimators` — number of trees (more is usually better, with diminishing returns).
- `max_features` — number of features considered per split ($\sqrt{d}$ for classification, $d/3$ for regression are common defaults).
- `max_depth`, `min_samples_leaf` — control individual tree complexity.

**Out-of-Bag (OOB) score** — each tree is trained on ~63% of data (bootstrap). The remaining ~37% (out-of-bag samples) serve as a free validation set. OOB score estimates generalization without a separate validation set.

In [ ]:
# Single tree vs bagging vs random forest
X_tr, X_te, y_tr, y_te = train_test_split(cancer.data, cancer.target, test_size=0.3, random_state=42)

models = {
    "Single Tree": DecisionTreeClassifier(random_state=42),
    "Bagging (50 trees)": BaggingClassifier(n_estimators=50, random_state=42),
    "Random Forest (100)": RandomForestClassifier(n_estimators=100, random_state=42),
}

print(f"{'Model':25s} {'Train Acc':>10s} {'Test Acc':>10s} {'CV Mean':>10s}")
print("-" * 57)
for name, model in models.items():
    model.fit(X_tr, y_tr)
    train_acc = accuracy_score(y_tr, model.predict(X_tr))
    test_acc = accuracy_score(y_te, model.predict(X_te))
    cv = cross_val_score(model, X_tr, y_tr, cv=5).mean()
    print(f"{name:25s} {train_acc:10.4f} {test_acc:10.4f} {cv:10.4f}")

### 4.3 Feature Importance

Random forests provide **feature importance** scores based on how much each feature decreases impurity (Gini or entropy) across all splits where it is used. More important features appear higher in more trees and produce larger impurity reductions.

Feature importance is useful for:
- Understanding which variables drive predictions.
- Feature selection (removing low-importance features).
- Communicating model behavior to stakeholders.

Caveat: importance scores are biased toward high-cardinality features. **Permutation importance** (shuffle one feature and measure performance drop) is a more reliable alternative.

In [ ]:
# Feature importance from random forest
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr, y_tr)

importances = pd.Series(rf.feature_importances_, index=cancer.feature_names).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
importances.plot(kind="barh", color="steelblue")
plt.xlabel("Feature Importance (Gini decrease)")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

---

## 5. Boosting

Boosting trains models **sequentially**, where each new model focuses on the mistakes of the previous ones. The final prediction is a weighted combination of all models.

### 5.1 AdaBoost

**Adaptive Boosting** adjusts sample weights after each round:

1. Train a weak learner on weighted data (all weights equal initially).
2. Increase weights of misclassified samples.
3. Train the next learner on reweighted data — it focuses on hard examples.
4. Combine all learners with weights proportional to their accuracy.

### 5.2 Gradient Boosting

Gradient boosting generalizes the idea: each new model fits the **residual errors** (negative gradients of the loss function) of the current ensemble.

For squared error loss, the residuals are simply $y_i - \hat{y}_i$ — the new tree learns to predict what the current ensemble got wrong.

The update at each step:

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

where $h_m$ is the new tree, and $\eta$ is the **learning rate** (shrinkage) — a regularization parameter that prevents each tree from contributing too much.

### 5.3 Modern Gradient Boosting Libraries

- **XGBoost** — regularized gradient boosting with efficient tree construction, parallel processing, and built-in handling of missing values.
- **LightGBM** — leaf-wise tree growth (grows the leaf with the largest loss reduction), faster on large datasets.
- **CatBoost** — handles categorical features natively, reduces overfitting through ordered boosting.

These libraries dominate tabular data competitions and production systems.

In [ ]:
# Boosting: AdaBoost and Gradient Boosting
models_boost = {
    "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=42),
    "AdaBoost (50)": AdaBoostClassifier(n_estimators=50, random_state=42),
    "Gradient Boosting (100)": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                                             max_depth=3, random_state=42),
    "Random Forest (100)": RandomForestClassifier(n_estimators=100, random_state=42),
}

print(f"{'Model':30s} {'Test Acc':>10s} {'CV Mean':>10s}")
print("-" * 52)
for name, model in models_boost.items():
    model.fit(X_tr, y_tr)
    test_acc = accuracy_score(y_te, model.predict(X_te))
    cv = cross_val_score(model, X_tr, y_tr, cv=5).mean()
    print(f"{name:30s} {test_acc:10.4f} {cv:10.4f}")

In [ ]:
# Learning rate effect in gradient boosting
learning_rates = [0.01, 0.05, 0.1, 0.3, 1.0]
train_scores, test_scores = [], []

for lr in learning_rates:
    gb = GradientBoostingClassifier(n_estimators=100, learning_rate=lr, max_depth=3, random_state=42)
    gb.fit(X_tr, y_tr)
    train_scores.append(accuracy_score(y_tr, gb.predict(X_tr)))
    test_scores.append(accuracy_score(y_te, gb.predict(X_te)))

plt.figure(figsize=(8, 5))
x = range(len(learning_rates))
plt.plot(x, train_scores, "b-o", label="Train", linewidth=2)
plt.plot(x, test_scores, "r-o", label="Test", linewidth=2)
plt.xticks(x, [str(lr) for lr in learning_rates])
plt.xlabel("Learning rate")
plt.ylabel("Accuracy")
plt.title("Gradient Boosting: lower learning rate often generalizes better")
plt.legend()
plt.show()

---

## 6. Bagging vs Boosting

| Aspect | Bagging (Random Forest) | Boosting (Gradient Boosting) |
|--------|------------------------|------------------------------|
| Training | Parallel, independent | Sequential, dependent |
| Focus | Reduce variance | Reduce bias |
| Base learners | Complex (deep trees) | Simple (shallow trees) |
| Overfitting risk | Low (averaging helps) | Higher (must tune carefully) |
| Speed | Fast to train (parallelizable) | Slower (sequential) |
| Hyperparameters | Fewer, more forgiving | More sensitive (learning rate, depth, estimators) |
| Outliers | Robust (averaging) | Sensitive (focuses on hard examples) |

In practice, both are strong. Random forests are easier to tune and harder to overfit. Gradient boosting often achieves slightly higher accuracy with careful tuning. Try both and compare with cross-validation.

---

## 7. Regularization: The Core Idea

Regularization prevents overfitting by adding a **penalty** for model complexity. Instead of minimizing only prediction error, the model minimizes error plus a complexity term:

$$\text{Loss}_{\text{total}} = \text{Prediction Loss} + \alpha \cdot \text{Complexity Penalty}$$

The hyperparameter $\alpha$ (or $\lambda$) controls the tradeoff. $\alpha = 0$ is no regularization. Larger $\alpha$ means simpler models.

Regularization works because simpler models are less likely to fit noise. By constraining the model, we sacrifice some training accuracy for better generalization.

Regularization appears in many forms:
- **Linear models** — L1 (Lasso), L2 (Ridge), Elastic Net.
- **Trees** — max depth, min samples, pruning.
- **Boosting** — learning rate, number of estimators, subsampling.
- **Neural networks** — weight decay, dropout, early stopping.

### 7.1 L2 Regularization (Ridge)

Adds a penalty proportional to the sum of squared weights:

$$\text{Loss} = \text{MSE} + \alpha \sum_{j} w_j^2$$

Shrinks all weights toward zero but rarely to exactly zero. Produces models where all features contribute slightly. Good when many features are mildly relevant.

### 7.2 L1 Regularization (Lasso)

Adds a penalty proportional to the sum of absolute weights:

$$\text{Loss} = \text{MSE} + \alpha \sum_{j} |w_j|$$

Can shrink some weights to exactly zero — performing **automatic feature selection**. Good when only a few features are truly important.

### 7.3 Elastic Net

Combines L1 and L2 penalties:

$$\text{Loss} = \text{MSE} + \alpha \left( \rho \sum_{j} |w_j| + \frac{1-\rho}{2} \sum_{j} w_j^2 \right)$$

Balances feature selection (Lasso) with handling correlated features (Ridge).

In [ ]:
# Regularization paths: Ridge vs Lasso
X_reg, y_reg = make_regression(n_samples=100, n_features=20, n_informative=5, noise=10, random_state=42)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)
X_tr_r = StandardScaler().fit_transform(X_tr_r)
X_te_r = StandardScaler().fit_transform(X_te_r)

alphas = np.logspace(-2, 4, 50)
ridge_coefs, lasso_coefs = [], []
ridge_test, lasso_test = [], []

for a in alphas:
    ridge = Ridge(alpha=a).fit(X_tr_r, y_tr_r)
    lasso = Lasso(alpha=a, max_iter=5000).fit(X_tr_r, y_tr_r)
    ridge_coefs.append(ridge.coef_)
    lasso_coefs.append(lasso.coef_)
    ridge_test.append(mean_squared_error(y_te_r, ridge.predict(X_te_r)))
    lasso_test.append(mean_squared_error(y_te_r, lasso.predict(X_te_r)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(alphas, np.array(ridge_coefs), alpha=0.7)
axes[0].set_xscale("log")
axes[0].set_xlabel("α")
axes[0].set_ylabel("Coefficient value")
axes[0].set_title("Ridge: coefficients shrink but stay non-zero")

axes[1].plot(alphas, np.array(lasso_coefs), alpha=0.7)
axes[1].set_xscale("log")
axes[1].set_xlabel("α")
axes[1].set_title("Lasso: coefficients shrink to exactly zero")

axes[2].plot(alphas, ridge_test, "b-", label="Ridge", linewidth=2)
axes[2].plot(alphas, lasso_test, "r-", label="Lasso", linewidth=2)
axes[2].set_xscale("log")
axes[2].set_xlabel("α")
axes[2].set_ylabel("Test MSE")
axes[2].set_title("Test error vs regularization strength")
axes[2].legend()

plt.tight_layout()
plt.show()

### 7.4 Polynomial Features and Regularization

Polynomial features increase model capacity — a linear model with degree-10 polynomial features can fit complex curves. Without regularization, this overfits badly. Ridge or Lasso on polynomial features balances flexibility with constraint.

This combination — expanding features then regularizing — is a powerful technique for nonlinear problems while maintaining the interpretability and speed of linear models.

In [ ]:
# Polynomial regression: overfitting vs regularized
X_1d = np.linspace(0, 1, 30).reshape(-1, 1)
y_1d = np.sin(2 * np.pi * X_1d.ravel()) + np.random.normal(0, 0.15, 30)
X_plot = np.linspace(0, 1, 200).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

configs = [
    (Pipeline([("poly", PolynomialFeatures(15)), ("lr", LinearRegression())]), "Degree 15, no reg"),
    (Pipeline([("poly", PolynomialFeatures(15)), ("lr", Ridge(alpha=0.01))]), "Degree 15, Ridge α=0.01"),
    (Pipeline([("poly", PolynomialFeatures(15)), ("lr", Lasso(alpha=0.001))]), "Degree 15, Lasso α=0.001"),
]

for ax, (pipe, title) in zip(axes, configs):
    pipe.fit(X_1d, y_1d)
    ax.scatter(X_1d, y_1d, alpha=0.6, s=20)
    ax.plot(X_plot, pipe.predict(X_plot), "r-", linewidth=2)
    r2 = r2_score(y_1d, pipe.predict(X_1d))
    ax.set_title(f"{title}\nR²={r2:.3f}")

plt.tight_layout()
plt.show()

---

## 8. Regularization in Tree-Based Models

Trees and ensembles have their own regularization mechanisms beyond the linear model penalties:

**Decision trees:**
- `max_depth` — limit tree height.
- `min_samples_split`, `min_samples_leaf` — require minimum data per node/leaf.
- `max_leaf_nodes` — cap the number of leaves.
- `ccp_alpha` — cost-complexity pruning.

**Random forests:**
- `n_estimators` — more trees reduce variance (but with diminishing returns).
- `max_features` — fewer features per split increases tree diversity.
- `max_samples` — bootstrap sample size (less than 1.0 adds randomness).

**Gradient boosting:**
- `learning_rate` ($\eta$) — shrink each tree's contribution.
- `n_estimators` — more trees with lower learning rate generalizes better.
- `max_depth` — shallow trees (stumps or depth 3–6) prevent overfitting.
- `subsample` — train each tree on a random fraction of data (stochastic gradient boosting).
- `min_samples_leaf` — minimum samples in leaves.

In [ ]:
# Gradient boosting: n_estimators vs learning_rate tradeoff
configs = [
    (50, 0.3, "50 trees, lr=0.3"),
    (100, 0.1, "100 trees, lr=0.1"),
    (300, 0.05, "300 trees, lr=0.05"),
    (500, 0.01, "500 trees, lr=0.01"),
]

print(f"{'Config':25s} {'Train Acc':>10s} {'Test Acc':>10s}")
print("-" * 47)
for n_est, lr, label in configs:
    gb = GradientBoostingClassifier(n_estimators=n_est, learning_rate=lr,
                                     max_depth=3, random_state=42).fit(X_tr, y_tr)
    print(f"{label:25s} {accuracy_score(y_tr, gb.predict(X_tr)):10.4f} "
          f"{accuracy_score(y_te, gb.predict(X_te)):10.4f}")

---

## 9. Early Stopping

Early stopping is a regularization technique that halts training when validation performance stops improving. Instead of training for a fixed number of iterations, monitor a validation metric and stop when it plateaus or worsens.

This applies to:
- **Gradient boosting** — stop adding trees when validation error increases.
- **Neural networks** — stop training epochs when validation loss rises.
- **Any iterative algorithm** — with a validation set to monitor.

Early stopping is elegant because it requires no explicit complexity penalty — the model simply stops before it overfits. It is one of the most practical regularization methods in production.

---

## 10. Choosing the Right Approach

| Situation | Recommended Approach |
|-----------|---------------------|
| Need interpretability | Single decision tree (shallow) |
| Tabular data, general purpose | Random forest or gradient boosting |
| Many features, few relevant | Lasso or gradient boosting with feature importance |
| Correlated features | Ridge or Elastic Net |
| Small dataset | Regularized linear model or shallow tree |
| Large dataset, accuracy critical | Gradient boosting (XGBoost, LightGBM) with tuning |
| Fast training required | Random forest (parallelizable) |
| Nonlinear with few features | Polynomial features + Ridge |

A practical workflow: start with a regularized baseline (Ridge or logistic regression), then try random forest, then gradient boosting. Compare with cross-validation. Use the simplest model that meets performance requirements.

In [ ]:
# Final comparison across all approaches
X_all, y_all = cancer.data, cancer.target

final_models = {
    "Logistic Regression": Pipeline([("scaler", StandardScaler()),
                                     ("clf", LogisticRegression(C=1.0, max_iter=1000))]),
    "Decision Tree (depth=4)": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                       max_depth=3, subsample=0.8, random_state=42),
}

print(f"{'Model':25s} {'CV Mean':>10s} {'CV Std':>10s}")
print("-" * 47)
for name, model in final_models.items():
    scores = cross_val_score(model, X_all, y_all, cv=5, scoring="accuracy")
    print(f"{name:25s} {scores.mean():10.4f} {scores.std():10.4f}")

---

## 11. Summary

**Decision trees** make predictions through interpretable sequences of feature tests. Unchecked, they overfit — controlled through depth limits, minimum samples, and pruning. **Ensemble methods** combine multiple trees for stronger predictions: **bagging** (random forests) reduces variance by averaging independent models; **boosting** (gradient boosting) reduces bias by sequentially correcting errors.

**Regularization** prevents overfitting across all model types. L1 (Lasso) and L2 (Ridge) penalize large weights in linear models. Trees regularize through depth and sample constraints. Boosting regularizes through learning rate and subsampling. Early stopping halts training before overfitting begins.

Random forests and gradient boosting remain the most reliable tools for tabular data. Understanding their mechanics, hyperparameters, and regularization options enables you to build models that are both accurate and robust.